# G1 and G3 Generation: Remaining Reference-Complete Test Cohort

## Goal and scope
Demonstrate direct generation (G1) and observed-human-analysis-mediated generation (G3) on the 16 remaining test-split documents with a unique teacher-educator feedback reference. The R1 six-document pilot is kept as a separate cohort. G2 consumes predicted analysis through the same request and output contracts.

## Conditions
- **G1: Direct baseline.** Provide ordered reflection segments and the rubric, with no segment analysis.
- **G2: Predicted-analysis intervention.** Provide the same reflection and rubric plus predicted scope and component-band labels.
- **G3: Observed human analysis.** Provide the same reflection and rubric plus available candidate-grain human annotations. These may be incomplete, ambiguous, or marked for review; this development condition is not an oracle.

Use the same generator model, shared prompt content, output schema, and generation settings across conditions. Keep reflection text, annotations, model outputs, and traces within approved protected storage.

## 1. Select the remaining reference-complete test cohort

The shared cohort selector selects test-split documents outside R1 that have exactly one nonempty teacher-feedback record. The expected cohort size is pinned in `config/experiment.toml`; missing or duplicate reference records fail closed.

### Load the configured cohort documents

The selector joins eligible teacher-feedback records with documents in the canonical test modeling split. The loader validates reflection and candidate-segment consistency but retains review-marked and ambiguous human annotations for G3.

In [1]:
from pathlib import Path
import importlib

from reflection_assessment_feedback import EXPERT_RUBRIC
import reflection_assessment_feedback.data as reflection_data
from reflection_assessment_feedback.development_cohorts import load_remaining_test_feedback_cohort
from reflection_assessment_feedback.experiment_config import load_experiment_config

reflection_data = importlib.reload(reflection_data)
load_human_analysis_context = reflection_data.load_human_analysis_context

PROJECT_ROOT = Path.cwd().resolve()
assert PROJECT_ROOT.name == "prebi_v1", "Run this notebook from the prebi_v1 project root."
EXPERIMENT_CONFIG = load_experiment_config()
DATASET_CONFIG = EXPERIMENT_CONFIG["dataset"]
DEVELOPMENT_COHORT = DATASET_CONFIG["generation_cohort"]
DATA_PATH = (
    PROJECT_ROOT
    / EXPERIMENT_CONFIG["paths"]["provisional_split_directory"]
    / f"provisional_{DATASET_CONFIG['prediction_split']}_modeling_wide.csv"
)
DEVELOPMENT_FEEDBACK = load_remaining_test_feedback_cohort()

DEVELOPMENT_BATCH = []
for feedback_record in DEVELOPMENT_FEEDBACK.to_dict(orient="records"):
    document, human_context = load_human_analysis_context(
        DATA_PATH,
        feedback_record["document_id"],
    )
    DEVELOPMENT_BATCH.append(
        {
            "document": document,
            "human_context": human_context,
            "human_feedback_record": feedback_record,
        }
    )
if len(DEVELOPMENT_BATCH) != DATASET_CONFIG["expected_generation_document_count"]:
    raise ValueError("Loaded G1/G3 batch size differs from the configured cohort size.")
print(
    f"Loaded {len(DEVELOPMENT_BATCH)} documents for {DEVELOPMENT_COHORT}; "
    f"segments={sum(len(item['document'].segments) for item in DEVELOPMENT_BATCH)}, "
    f"candidate bundles={sum(sum(len(segment.candidates) for segment in item['human_context'].segments) for item in DEVELOPMENT_BATCH)}."
)

Loaded 16 documents for remaining-test-reference-complete-v1; segments=567, candidate bundles=568.


### 2. Check the batch contents

The next cell reports aggregate document, segment, candidate-annotation, and review-flag counts only. It does not display reflection text, candidate labels, or identifiers.

In [2]:
candidate_bundle_count = sum(
    len(segment.candidates)
    for item in DEVELOPMENT_BATCH
    for segment in item["human_context"].segments
)
reviewed_segment_count = sum(
    any(candidate.requires_review for candidate in segment.candidates)
    for item in DEVELOPMENT_BATCH
    for segment in item["human_context"].segments
)
print(
    f"Development cohort: {len(DEVELOPMENT_BATCH)} documents, "
    f"{sum(len(item['document'].segments) for item in DEVELOPMENT_BATCH)} segments, "
    f"{candidate_bundle_count} candidate annotations, "
    f"{reviewed_segment_count} segments with review flags."
)

Development cohort: 16 documents, 567 segments, 568 candidate annotations, 1 segments with review flags.


### 3. Tracing policy

The shared `config/experiment.toml` enables LangSmith tracing for this approved batch. Confirm EU project access and retention approval before processing; traces may contain complete reflections, prompts, candidate analysis, and generated outputs. Provider approval remains a separate runtime gate.

In [3]:
from reflection_assessment_feedback.experiment_config import load_experiment_config

EXPERIMENT_CONFIG = load_experiment_config()
ENABLE_LANGSMITH_TRACING = EXPERIMENT_CONFIG["generation"]["enable_langsmith_tracing"]

### 4. Generate paired G1/G3 runs

After confirming provider approval, this cell runs one G1 and one G3 generation for each selected document. It retains completed conditions in kernel state so rerunning after an interrupted request skips finished calls. G3 includes observed candidate annotations, including review flags and unknowns; it is not an oracle condition.

In [7]:
import importlib
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv

from reflection_assessment_feedback.experiment_config import (
    experiment_config_sha256,
    load_experiment_config,
)
from reflection_assessment_feedback.prompt_registry import resolve_prompt
from reflection_assessment_feedback.rate_limit import reserve_request_slot

EXPERIMENT_CONFIG = load_experiment_config()
GENERATION_CONFIG = EXPERIMENT_CONFIG["generation"]
GENERATION_PROMPT = resolve_prompt(
    GENERATION_CONFIG["prompt_id"],
    GENERATION_CONFIG["expected_prompt_version"],
)
langsmith_dotenv_path = PROJECT_ROOT / ".env"
langsmith_dotenv = dotenv_values(langsmith_dotenv_path)
load_dotenv(langsmith_dotenv_path, override=False)
conflicting_langsmith_settings = [
    name
    for name in ("LANGSMITH_API_KEY", "LANGSMITH_PROJECT", "LANGSMITH_ENDPOINT")
    if langsmith_dotenv.get(name)
    and os.getenv(name)
    and langsmith_dotenv[name] != os.getenv(name)
]
if conflicting_langsmith_settings:
    raise RuntimeError(
        "LangSmith settings differ between .env and the active kernel environment: "
        f"{', '.join(conflicting_langsmith_settings)}. The kernel environment takes "
        "precedence; update or unset the conflicting value and restart the kernel. "
        "Secret values are not displayed."
    )

from langchain_google_genai import ChatGoogleGenerativeAI
import reflection_assessment_feedback.generator as generation_module
from reflection_assessment_feedback import make_generation_request

generation_module = importlib.reload(generation_module)
RubricGenerationRunner = generation_module.RubricGenerationRunner
if generation_module.PROMPT_VERSION != GENERATION_PROMPT.version:
    raise ValueError("Runtime generator prompt differs from the selected prompt artifact.")
if EXPERT_RUBRIC.version != GENERATION_CONFIG["expected_rubric_version"]:
    raise ValueError("Rubric version differs from experiment config.")

APPROVE_EXTERNAL_PROCESSING = True
REPETITIONS = GENERATION_CONFIG["repetitions"]
assert APPROVE_EXTERNAL_PROCESSING, "Confirm provider processing is approved before running."
assert os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"), (
    "Configure GOOGLE_API_KEY or GEMINI_API_KEY in the approved environment."
)
if ENABLE_LANGSMITH_TRACING:
    assert os.getenv("LANGSMITH_API_KEY"), (
        "Configure LANGSMITH_API_KEY in the local .env or approved environment."
    )
    assert os.getenv("LANGSMITH_PROJECT"), (
        "Configure LANGSMITH_PROJECT in the local .env or approved environment."
    )
    assert os.getenv("LANGSMITH_ENDPOINT", "https://eu.api.smith.langchain.com").rstrip("/") == (
        "https://eu.api.smith.langchain.com"
    ), "Tracing is restricted to the LangSmith EU endpoint."

protected_root_value = os.getenv("PREBI_DATA_ROOT")
if not protected_root_value:
    raise RuntimeError("Set PREBI_DATA_ROOT to approved protected storage before running the batch.")
protected_root = Path(protected_root_value).expanduser().resolve()
if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
    raise ValueError("PREBI_DATA_ROOT must be outside the project source tree.")
rate_limit_state = protected_root / ".rate_limits" / "gemini.state"

chat_model = ChatGoogleGenerativeAI(
    model=GENERATION_CONFIG["model_name"],
    temperature=GENERATION_CONFIG["temperature"],
)
runner = RubricGenerationRunner(
    chat_model,
    enable_langsmith_tracing=ENABLE_LANGSMITH_TRACING,
)
RUNS_BY_DOCUMENT = globals().get("RUNS_BY_DOCUMENT", {})
for batch_item in DEVELOPMENT_BATCH:
    batch_document = batch_item["document"]
    condition_runs = RUNS_BY_DOCUMENT.setdefault(
        batch_document.document_id,
        {"G1": [], "G3": []},
    )
    if set(condition_runs) != {"G1", "G3"}:
        raise ValueError("In-memory partial runs have unexpected conditions.")
    g1_request = make_generation_request(
        condition="G1",
        document=batch_document,
        rubric=EXPERT_RUBRIC,
    )
    g3_request = make_generation_request(
        condition="G3",
        document=batch_document,
        rubric=EXPERT_RUBRIC,
        analysis=batch_item["human_context"],
    )
    for condition, request in (("G1", g1_request), ("G3", g3_request)):
        completed_repetitions = len(condition_runs[condition])
        if completed_repetitions > REPETITIONS:
            raise ValueError("In-memory repetitions exceed the configured run count.")
        for repetition in range(completed_repetitions + 1, REPETITIONS + 1):
            reserve_request_slot(
                rate_limit_state,
                GENERATION_CONFIG["minimum_request_interval_seconds"],
            )
            condition_runs[condition].append(
                runner.generate(request, repetition=repetition)
            )

if set(RUNS_BY_DOCUMENT) != {item["document"].document_id for item in DEVELOPMENT_BATCH}:
    raise ValueError("In-memory G1/G3 runs do not match the selected cohort.")
if any(
    len(condition_runs[condition]) != REPETITIONS
    for condition_runs in RUNS_BY_DOCUMENT.values()
    for condition in ("G1", "G3")
):
    raise ValueError("Some in-memory G1/G3 conditions are incomplete; completed calls are retained for retry.")
print(
    f"G1/G3 complete for {len(RUNS_BY_DOCUMENT)} documents in {DEVELOPMENT_COHORT}; "
    f"Gemini request slots spaced by at least "
    f"{GENERATION_CONFIG['minimum_request_interval_seconds']} seconds."
)
if ENABLE_LANGSMITH_TRACING:
    print(
        "LangSmith tracing targeted EU endpoint "
        f"for project {os.environ['LANGSMITH_PROJECT']}; check warnings above for upload failures."
    )

/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_gen

G1/G3 complete for 16 documents in remaining-test-reference-complete-v1; Gemini request slots spaced by at least 7.5 seconds.
LangSmith tracing targeted EU endpoint for project prebi_v1; check warnings above for upload failures.


In [6]:
condition_run_counts = {
    condition: sum(
        len(runs.get(condition, []))
        for runs in RUNS_BY_DOCUMENT.values()
    )
    for condition in ("G1", "G3")
}
incomplete_document_count = sum(
    set(runs) != {"G1", "G3"}
    or any(len(runs[condition]) != REPETITIONS for condition in ("G1", "G3"))
    for runs in RUNS_BY_DOCUMENT.values()
)
print({
    "documents_started": len(RUNS_BY_DOCUMENT),
    "condition_runs": condition_run_counts,
    "documents_incomplete": incomplete_document_count,
})

{'documents_started': 1, 'condition_runs': {'G1': 1, 'G3': 0}, 'documents_incomplete': 1}


### 5. Save paired runs per document

Each document gets a separate protected G1/G3 comparison artifact under `PREBI_DATA_ROOT/g1-g3/demo-runs/`. Artifacts retain the exact G3 candidate-grain human context, review/missingness counts, rubric and prompt provenance, and the experiment-config fingerprint. Writes are atomic and use restrictive file permissions.

In [8]:
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from reflection_assessment_feedback.experiment_config import experiment_config_sha256
from reflection_assessment_feedback.generator import output_for_review
from reflection_assessment_feedback.prompt_registry import resolve_prompt
from reflection_assessment_feedback.rubric import RUBRIC_ARTIFACT

PROJECT_ROOT = Path.cwd().resolve()
assert PROJECT_ROOT.name == "prebi_v1", "Run this notebook from the prebi_v1 project root."
load_dotenv(PROJECT_ROOT / ".env", override=False)
protected_root_value = os.getenv("PREBI_DATA_ROOT")
if not protected_root_value:
    raise RuntimeError("Set PREBI_DATA_ROOT to approved protected storage before saving demo runs.")
protected_root = Path(protected_root_value).expanduser()
if not protected_root.is_absolute():
    raise ValueError("PREBI_DATA_ROOT must be an absolute path.")
protected_root = protected_root.resolve()
if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
    raise ValueError("PREBI_DATA_ROOT must be outside the project source tree.")

run_directory = protected_root / "g1-g3" / "demo-runs"
run_directory.mkdir(parents=True, exist_ok=True, mode=0o700)
if not run_directory.resolve().is_relative_to(protected_root):
    raise ValueError("Demo-run storage must resolve within PREBI_DATA_ROOT.")
expected_document_ids = {item["document"].document_id for item in DEVELOPMENT_BATCH}
if set(RUNS_BY_DOCUMENT) != expected_document_ids:
    raise ValueError("Generated G1/G3 documents do not match the selected development cohort.")


def serialize_run(run, batch_document):
    prompt_artifact = resolve_prompt(GENERATION_CONFIG["prompt_id"], run.prompt_version)
    if prompt_artifact.sha256 != run.prompt_artifact_sha256:
        raise ValueError("Run prompt artifact hash does not match the registered prompt version.")
    if run.rubric_artifact_sha256 != RUBRIC_ARTIFACT.sha256:
        raise ValueError("Run rubric hash does not match the registered rubric version.")
    return {
        "run_id": run.run_id,
        "condition": run.condition,
        "repetition": run.repetition,
        "model_name": run.model_name,
        "generation_parameters": run.generation_parameters,
        "prompt_version": run.prompt_version,
        "prompt_artifact_sha256": run.prompt_artifact_sha256,
        "prompt_components": run.prompt_components,
        "rendered_prompt_sha256": run.rendered_prompt_sha256,
        "rubric_artifact_sha256": run.rubric_artifact_sha256,
        "output": output_for_review(run.output, batch_document),
        "feedback_letter": run.feedback_letter,
    }


SAVED_COMPARISONS = []
for batch_item in DEVELOPMENT_BATCH:
    batch_document = batch_item["document"]
    condition_runs = RUNS_BY_DOCUMENT[batch_document.document_id]
    if set(condition_runs) != {"G1", "G3"}:
        raise ValueError("Each development document must have paired G1 and G3 runs.")
    if any(len(condition_runs[name]) != REPETITIONS for name in ("G1", "G3")):
        raise ValueError("Each condition must contain exactly REPETITIONS generated runs.")
    runs_by_repetition = {
        condition: {run.repetition: run for run in runs}
        for condition, runs in condition_runs.items()
    }
    if set(runs_by_repetition["G1"]) != set(runs_by_repetition["G3"]):
        raise ValueError("G1 and G3 runs must have matching repetition numbers.")

    human_context = batch_item["human_context"]
    candidate_count = sum(len(segment.candidates) for segment in human_context.segments)
    reviewed_segment_count = sum(
        any(candidate.requires_review for candidate in segment.candidates)
        for segment in human_context.segments
    )
    multi_candidate_segment_count = sum(
        len(segment.candidates) > 1 for segment in human_context.segments
    )
    comparison_package = {
        "schema_version": "1.5",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "experiment_config_sha256": experiment_config_sha256(),
        "development_cohort": DEVELOPMENT_COHORT,
        "document_id": batch_document.document_id,
        "segment_count": len(batch_document.segments),
        "rubric_id": EXPERT_RUBRIC.rubric_id,
        "rubric_version": EXPERT_RUBRIC.version,
        "rubric_artifact_sha256": RUBRIC_ARTIFACT.sha256,
        "g3_analysis_quality": {
            "kind": "observed_human_annotations",
            "candidate_annotation_count": candidate_count,
            "reviewed_segment_count": reviewed_segment_count,
            "multiple_candidate_segment_count": multi_candidate_segment_count,
        },
        "g3_human_analysis_context": human_context.model_dump(mode="json"),
        "pairs": [
            {
                "repetition": repetition,
                "G1": serialize_run(runs_by_repetition["G1"][repetition], batch_document),
                "G3": serialize_run(runs_by_repetition["G3"][repetition], batch_document),
            }
            for repetition in sorted(runs_by_repetition["G1"])
        ],
    }
    output_path = run_directory / (
        f"comparison-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid4().hex}.json"
    )
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=run_directory,
            prefix=".comparison-", suffix=".tmp", delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)
            json.dump(comparison_package, temporary_file, ensure_ascii=False, indent=2)
            temporary_file.write("\n")
        os.chmod(temporary_path, 0o600)
        os.replace(temporary_path, output_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)
    SAVED_COMPARISONS.append((output_path, comparison_package))

print(f"Saved paired G1/G3 artifacts for {len(SAVED_COMPARISONS)} documents in {DEVELOPMENT_COHORT}.")

Saved paired G1/G3 artifacts for 16 documents in remaining-test-reference-complete-v1.


In [9]:
if len(SAVED_COMPARISONS) != DATASET_CONFIG["expected_generation_document_count"]:
    raise ValueError("Not every configured cohort document has a saved G1/G3 comparison.")
items_by_id = {item["document"].document_id: item for item in DEVELOPMENT_BATCH}
for comparison_path, saved_comparison in SAVED_COMPARISONS:
    batch_item = items_by_id[saved_comparison["document_id"]]
    batch_document = batch_item["document"]
    human_context = batch_item["human_context"]
    assert saved_comparison["schema_version"] == "1.5"
    assert saved_comparison["development_cohort"] == DEVELOPMENT_COHORT
    assert saved_comparison["experiment_config_sha256"] == experiment_config_sha256()
    assert saved_comparison["rubric_artifact_sha256"] == RUBRIC_ARTIFACT.sha256
    assert saved_comparison["segment_count"] == len(batch_document.segments)
    assert saved_comparison["g3_human_analysis_context"] == human_context.model_dump(mode="json")
    assert len(saved_comparison["pairs"]) == REPETITIONS
    orders_by_id = {segment.segment_id: segment.order for segment in batch_document.segments}
    for pair in saved_comparison["pairs"]:
        assert pair["G1"]["condition"] == "G1"
        assert pair["G3"]["condition"] == "G3"
        assert pair["G1"]["repetition"] == pair["G3"]["repetition"] == pair["repetition"]
        for condition in ("G1", "G3"):
            run_data = pair[condition]
            assert len(run_data["prompt_artifact_sha256"]) == 64
            assert len(run_data["rendered_prompt_sha256"]) == 64
            assert len(run_data["rubric_artifact_sha256"]) == 64
            assert len(run_data["prompt_components"]) == (2 if condition == "G3" else 1)
            for assessment in run_data["output"]["assessment"]["component_assessments"]:
                assert assessment["evidence_segment_orders"] == [
                    orders_by_id[segment_id] for segment_id in assessment["evidence_segment_ids"]
                ]
    disk_copy = json.loads(comparison_path.read_text(encoding="utf-8"))
    assert disk_copy == saved_comparison
print("All remaining-cohort G1/G3 artifacts passed pairing, candidate-context, evidence, prompt-component, and config checks.")

All remaining-cohort G1/G3 artifacts passed pairing, candidate-context, evidence, prompt-component, and config checks.


### 6. Verify batch persistence

The final cell reports aggregate counts only. Full reflection text, human feedback, candidate annotations, generated outputs, and traces remain in approved protected storage.

In [8]:
print(
    {
        "development_documents": len(SAVED_COMPARISONS),
        "segments": sum(len(item["document"].segments) for item in DEVELOPMENT_BATCH),
        "candidate_annotations": sum(
            len(segment.candidates)
            for item in DEVELOPMENT_BATCH
            for segment in item["human_context"].segments
        ),
        "paired_g1_g3_runs": sum(
            len(package["pairs"]) for _, package in SAVED_COMPARISONS
        ),
        "artifacts_validated": len(SAVED_COMPARISONS),
    }
)

{'development_documents': 6, 'segments': 264, 'candidate_annotations': 265, 'paired_g1_g3_runs': 6, 'artifacts_validated': 6}
